In [1]:
# Google Colab drive mount disabled for local execution
print('Running locally. Using local relative paths.')


Running locally. Using local relative paths.


In [2]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import tensorflow as tf
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

I0000 00:00:1781193161.566361   12095 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781193161.569016   12095 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781193161.674182   12095 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1781193165.580852   12095 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781193165.582313   12095 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
data = np.load("../processed/full_weather_9features_v2.npz", allow_pickle=True)

X_seq_full = data["X_seq_full"]
meta = data["meta"]

print(X_seq_full.shape)

(17927, 12, 12)


In [4]:
selected_idx = [0,1,2,3,4,5,6,11]

In [5]:
selected_features_idx = [0, 1, 2, 3, 4, 5, 6, 11]
X_corn = X_seq_full[:, 3:9, selected_features_idx]
print(X_corn.shape)


(17927, 6, 8)


In [6]:
from tensorflow.keras.layers import Input
seq_input = Input(shape=(6, 8))

In [7]:
YEARS = ['2017','2018','2019','2020','2021','2022']

In [8]:
import pandas as pd
yield_df = pd.read_csv("../processed/corn_all_years.csv")
yield_df.columns = ['fips', 'year', 'yield']
yield_df['fips'] = yield_df['fips'].astype(str).str.zfill(5)
yield_df['year'] = yield_df['year'].astype(str)
yield_df['yield'] = pd.to_numeric(yield_df['yield'], errors='coerce')
yield_df = yield_df.dropna(subset=['yield'])


In [11]:
print(yield_df.head())
print("Shape:", yield_df.shape)
print("Unique FIPS:", yield_df['fips'].nunique())

    fips  year  yield
0  01003  2021  154.0
1  01005  2021  188.8
2  01009  2021  151.7
3  01015  2021  160.2
4  01019  2021  173.3
Shape: (10246, 3)
Unique FIPS: 1964


In [12]:
final_X = []
final_y = []
final_meta = []

for i, (fips, year) in enumerate(meta):

    match = yield_df[
        (yield_df['fips'] == fips) &
        (yield_df['year'] == year)
    ]

    if len(match) == 1:
        final_X.append(X_corn[i])
        final_y.append(match['yield'].values[0])
        final_meta.append((fips, year))

X = np.array(final_X)
y = np.array(final_y)

print(X.shape, y.shape)

(8375, 6, 8) (8375,)


In [13]:
# total rainfall (Apr–Sep)
total_rain = X[:, :, 4].sum(axis=1)

# avg temperature (use max+min)
avg_temp_season = X[:, :, 0:2].mean(axis=(1,2))

In [14]:
meta_df = pd.DataFrame(final_meta, columns=["fips", "year"])

train_idx = meta_df["year"] != "2022"
test_idx  = meta_df["year"] == "2022"

X_train = X[train_idx]
X_test  = X[test_idx]

y_train = y[train_idx]
y_test  = y[test_idx]


In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
fips_encoded = le.fit_transform(meta_df['fips'])

fips_train = fips_encoded[train_idx]
fips_test  = fips_encoded[test_idx]

In [16]:
from tensorflow.keras.layers import Embedding, Flatten

fips_input = Input(shape=(1,))

fips_emb = Embedding(input_dim=len(le.classes_), output_dim=8)(fips_input)
fips_emb = Flatten()(fips_emb)

E0000 00:00:1781193220.699777   12095 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_reshaped = X_train.reshape(-1, X_train.shape[2])
X_test_reshaped  = X_test.reshape(-1, X_test.shape[2])

X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled  = scaler.transform(X_test_reshaped)

X_train = X_train_scaled.reshape(X_train.shape)
X_test  = X_test_scaled.reshape(X_test.shape)

In [18]:
yield_map = {(row['fips'], str(row['year'])): row['yield'] for _, row in yield_df.iterrows()}

X_lag = []

for fips, year in final_meta:
    yr = int(year)
    y1 = yield_map.get((fips, str(yr - 1)), 0)
    y2 = yield_map.get((fips, str(yr - 2)), 0)
    
    X_lag.append([y1, y2])

X_lag = np.array(X_lag)

X_lag_train = X_lag[train_idx]
X_lag_test  = X_lag[test_idx]


In [19]:
yield_trend = X_lag[:, 0] - X_lag[:, 1]

In [20]:
X_extra = np.stack([total_rain, avg_temp_season, yield_trend], axis=1)

X_lag = np.concatenate([X_lag, X_extra], axis=1)

In [21]:
X_lag_train = X_lag[train_idx]
X_lag_test  = X_lag[test_idx]

In [22]:
lag_scaler = StandardScaler()

X_lag_train = lag_scaler.fit_transform(X_lag_train)
X_lag_test  = lag_scaler.transform(X_lag_test)

In [23]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(y_train.reshape(-1,1))
y_test  = y_scaler.transform(y_test.reshape(-1,1))

In [24]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam

# sequence input
seq_input = Input(shape=(6, 8)) # Changed shape from (6, 12) to (12, 7)
x = LSTM(96)(seq_input)
x = Dropout(0.2)(x)
x = Dense(32, activation='relu')(x)

# lag input
lag_input = Input(shape=(5,))
y_lag = Dense(16, activation='relu')(lag_input)

# combine
combined = Concatenate()([x, y_lag, fips_emb])

z = Dense(32, activation='relu')(combined)
output = Dense(1)(z)

model = Model(inputs=[seq_input, lag_input, fips_input], outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse',
    metrics=['mae']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 6, 8)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 96)        │     40,320 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 96)        │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │     14,680 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      3,104 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         96 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 56)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0],    │
│                     │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      1,824 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 60,057 (234.60 KB)

 Trainable params: 60,057 (234.60 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    [X_train, X_lag_train, fips_train], y_train,
    validation_data=([X_test, X_lag_test, fips_test], y_test),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/50



  1/109 ━━━━━━━━━━━━━━━━━━━━ 6:37 4s/step - loss: 1.7390 - mae: 1.0603


  6/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.4096 - mae: 0.9430


 12/109 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.3274 - mae: 0.9130


 18/109 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.2732 - mae: 0.8954

...


 90/109 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0913 - mae: 0.2317


 96/109 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0912 - mae: 0.2314


102/109 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0910 - mae: 0.2311


108/109 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0909 - mae: 0.2309


109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0885 - mae: 0.2263 - val_loss: 0.4320 - val_mae: 0.5198


In [26]:
from sklearn.metrics import r2_score

y_pred = model.predict([X_test, X_lag_test, fips_test])

y_pred = y_scaler.inverse_transform(y_pred)
y_test_actual = y_scaler.inverse_transform(y_test)

r2 = r2_score(y_test_actual, y_pred)

print("🔥 Final R2:", r2)


 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 276ms/step


14/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step   


27/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


40/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


🔥 Final R2: 0.6951564700421498


In [27]:
model.save("../corn_lstm_model.keras")

In [28]:
import joblib

# Save the y_scaler
joblib.dump(y_scaler, '../y_scaler.pkl')
print('y_scaler saved successfully.')

# Save the x_scaler
joblib.dump(scaler, '../x_scaler.pkl')
print('x_scaler saved successfully.')

# Save the lag_scaler
joblib.dump(lag_scaler, '../lag_scaler.pkl')
print('lag_scaler saved successfully.')

# Save the LabelEncoder for FIPS codes
joblib.dump(le, '../fips_label_encoder.pkl')
print('fips_label_encoder saved successfully.')

y_scaler saved successfully.
x_scaler saved successfully.
lag_scaler saved successfully.
fips_label_encoder saved successfully.


In [29]:
import pandas as pd

history_df = pd.DataFrame(history.history)
history_df.to_csv('../corn_model_training_history.csv', index=False)
print('Training history saved successfully.')

Training history saved successfully.


In [30]:
import json

config = {
    "crop": "corn",

    "months_used": [4, 5, 6, 7, 8, 9],

    "sequence_features": [
        {"index": 0, "name": "max_temp_C"},
        {"index": 1, "name": "min_temp_C"},
        {"index": 2, "name": "temp_range"},
        {"index": 3, "name": "gdd"},
        {"index": 4, "name": "precipitation"},
        {"index": 5, "name": "relative_humidity"},
        {"index": 6, "name": "radiation"}
    ],

    "engineered_features": {
        "total_rain": "sum(X[:, :, 4])",
        "avg_temp_season": "mean(X[:, :, 0:2])"
    },

    "lag_features": [
        "yield_lag_1",
        "yield_lag_2",
        "yield_trend",
        "total_rain",
        "avg_temp_season"
    ],

    "input_shape": {
        "sequence": [6, 7],
        "lag": 5
    }
}

with open('../corn_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("Config updated successfully.")

Config updated successfully.


In [31]:
fips_arr = np.array([f for f, _ in final_meta])
year_arr = np.array([y for _, y in final_meta])

total_rain = X[:, :, 4].sum(axis=1)
avg_temp_season = X[:, :, 0:2].mean(axis=(1,2))
yield_trend = X_lag[:, 0] - X_lag[:, 1]
rain_flowering = X[:, 2:4, 4].sum(axis=1)
heat_stress = (X[:, :, 0] > 35).sum(axis=1)
gdd_total = X[:, :, 3].sum(axis=1)
avg_humidity = X[:, :, 5].mean(axis=1)
et_total = X[:, :, 7].sum(axis=1)
X_extra_8 = np.stack([total_rain, avg_temp_season, yield_trend, rain_flowering, heat_stress, gdd_total, avg_humidity, et_total], axis=1)

np.savez("../corn4_data.npz",
         X=X,
         X_extra=X_extra_8,
         y=y,
         fips=fips_arr,
         year=year_arr)

In [32]:
print(X.shape)
print(fips_arr.shape)
print(year_arr.shape)

(8375, 6, 8)
(8375,)
(8375,)
